# LLM + Quantization + LoRA + SFT

In [ ]:
# !pip install bitsandbytes datasets trl peft huggingface_hub[hf_xet]

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from peft.utils.peft_types import TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

### Load Model and Tokenizer

- https://huggingface.co/t-tech/T-lite-it-1.0
- https://huggingface.co/docs/transformers/v4.51.3/quantization/overview

In [ ]:
model_name = "t-tech/T-lite-it-1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)

In [ ]:
model

### Usage Example

In [ ]:
input_text = "What is utionas拒"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    generation_kwargs = dict(
        inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id, temperature=0.1, do_sample=True
    )
    outputs = model.generate(**generation_kwargs)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
inputs

### Apply LoRA to the Model

https://huggingface.co/docs/peft/main/en/conceptual_guides/adapter

In [ ]:
list(TaskType)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
model

### Load SFT Dataset

https://huggingface.co/datasets/databricks/databricks-dolly-15k

Actually, we do not want to calculate the loss on the instructions. So you should fix masking

In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")


def preprocess_function(examples):
    inputs = [
        f"### Instruction:\n{ins}\n\n### Response:\n{res}"
        for ins, res in zip(examples["instruction"], examples["response"])
    ]
    return tokenizer(inputs, truncation=True, padding="max_length", max_length=512)


tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [ ]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.3)
tokenized_dataset

### Train Model

- https://huggingface.co/docs/trl/en/sft_trainer
- https://github.com/huggingface/trl/blob/c163cf5/trl/trainer/sft_trainer.py#L155

$$\text{Loss} = -\sum_{t=1}^{T} \log P(y_t | x, y_{<t})$$

where:
- $T$ — the length of the target sequence (e.g., the *response* in a dialogue)
- $y_t$ — the true token at position $t$
- $x$ — the input data (e.g., prompt or previous phrases in the dialog)
- $y_{<t}$ — all previous generated tokens before position $t$
- $P(y_t | x, y_{<t})$ — the probability of the token $y_t$ predicted by the model

In [ ]:
training_args = SFTConfig(
    output_dir="./sft_output",
    per_device_train_batch_size=4,
    max_steps=1000,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()